# Model Quality Monitoring Example

This is the third notebook solution. It shows how to monitor model quality separately from the automated data drift and data quality pipeline.

Model quality needs predictions and ground truth labels. Because labels are often delayed, this notebook is designed to run when those labels are available.

The example expects two explicit S3 file locations:

- `predictions_s3_uri`: CSV with predicted labels and optional probabilities
- `ground_truth_s3_uri`: CSV with true labels

The processor creates binary classification metrics and an Evidently `ClassificationPreset` report, then logs everything to SageMaker managed MLflow.


## Section 1: Install Packages and Import Libraries


In [ ]:
%pip install --no-cache-dir 'sagemaker>=3' mlflow==3.4.0 sagemaker-mlflow==0.2.0 pandas scikit-learn

In [ ]:
import boto3
from datetime import datetime

import pandas as pd
import sagemaker
from sagemaker.core import image_uris
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core.processing import ScriptProcessor
from sagemaker.core.shapes import ProcessingOutput, ProcessingS3Output
from sagemaker.core.workflow.parameters import ParameterString
from sagemaker.core.workflow.pipeline_context import PipelineSession
from sagemaker.mlops.workflow.pipeline import Pipeline
from sagemaker.mlops.workflow.steps import ProcessingStep

In [ ]:
sagemaker_session = Session()
pipeline_session = PipelineSession()

region = sagemaker_session.boto_region_name
role = get_execution_role()
bucket = sagemaker_session.default_bucket()
prefix = 'monitoring-pipeline'
timestamp_suffix = datetime.now().strftime('%Y%m%d%H%M%S')

sm_client = boto3.client('sagemaker', region_name=region)

print(f'Region: {region}')
print(f'Default S3 bucket: {bucket}')
print(f'SageMaker execution role: {role}')
print(f'Timestamp suffix: {timestamp_suffix}')

## Section 2: Configure Model Quality Inputs

Use files that already contain predictions and labels. The rows must describe the same records in the same order for this simple example. In a production system, join predictions and labels by a stable record ID before creating these CSV files.


In [ ]:
# ============================================================
# UPDATE THESE VALUES FOR YOUR ENVIRONMENT
# ============================================================

# Predictions CSV. Required column: prediction. Optional column: prediction_proba.
predictions_s3_uri = f's3://{bucket}/{prefix}/model-quality/predictions.csv'

# Ground truth CSV. Required column: target.
ground_truth_s3_uri = f's3://{bucket}/{prefix}/model-quality/ground_truth.csv'

# Column names expected by scripts/model_quality_processor.py.
target_column = 'target'
prediction_column = 'prediction'
prediction_proba_column = 'prediction_proba'
positive_label = '1'

# If you ran Notebook 1 and have local files, set this to True to create a small
# predictions/ground-truth pair from local demo outputs and upload it to S3.
upload_demo_files = False
local_batch_transform_predictions = 'data/predictions.csv'
local_test_labels = 'data/test.csv'

# MLflow configuration. Use the same experiment as training if you want all runs
# visible together, or use a separate experiment if your team prefers that.
mlflow_app_name = 'DefaultMLFlowApp'
mlflow_experiment_name = 'test-monitor-pipeline'
mlflow_run_name = f'model_quality_monitoring_{timestamp_suffix}'

pipeline_name = 'model-quality-monitoring-pipeline'
instance_type = 'ml.m5.large'

print(f'Predictions S3 URI       : {predictions_s3_uri}')
print(f'Ground truth S3 URI      : {ground_truth_s3_uri}')
print(f'MLflow app               : {mlflow_app_name}')
print(f'MLflow experiment        : {mlflow_experiment_name}')
print(f'Pipeline name            : {pipeline_name}')

In [ ]:
# Optional helper for local demo files created by Notebook 1.
# Batch Transform often writes a CSV with one probability per row and no header.
# This cell converts it into a clearer predictions CSV with headers.
if upload_demo_files:
    raw_predictions = pd.read_csv(
        local_batch_transform_predictions,
        header=None,
        names=[prediction_proba_column],
    )

    raw_predictions[prediction_column] = (
        raw_predictions[prediction_proba_column] > 0.5
    ).astype(int)

    predictions_for_quality = raw_predictions[[prediction_column, prediction_proba_column]]

    # data/test.csv from Notebook 1 uses the first column as the true label.
    labels = pd.read_csv(local_test_labels, header=None)
    ground_truth_for_quality = pd.DataFrame({target_column: labels.iloc[:, 0].values})

    predictions_for_quality.to_csv('data/model_quality_predictions.csv', index=False)
    ground_truth_for_quality.to_csv('data/model_quality_ground_truth.csv', index=False)

    predictions_s3_uri = sagemaker_session.upload_data(
        path='data/model_quality_predictions.csv',
        bucket=bucket,
        key_prefix=f'{prefix}/model-quality',
    )
    ground_truth_s3_uri = sagemaker_session.upload_data(
        path='data/model_quality_ground_truth.csv',
        bucket=bucket,
        key_prefix=f'{prefix}/model-quality',
    )

    print(f'Uploaded predictions to : {predictions_s3_uri}')
    print(f'Uploaded ground truth to: {ground_truth_s3_uri}')
else:
    print('Skipping demo upload. The pipeline will use the configured S3 URIs.')

## Section 3: Resolve the SageMaker Managed MLflow App ARN


In [ ]:
mlflow_apps = sm_client.list_mlflow_apps()

mlflow_app_arn = None
for app in mlflow_apps['Summaries']:
    if app['Name'] == mlflow_app_name:
        mlflow_app_arn = app['Arn']
        break

if mlflow_app_arn is None:
    available_apps = [app['Name'] for app in mlflow_apps['Summaries']]
    raise ValueError(f'MLflow app {mlflow_app_name} not found. Available apps: {available_apps}')

print(f'MLflow app name: {mlflow_app_name}')
print(f'MLflow app ARN : {mlflow_app_arn}')

## Section 4: Define the Model Quality Pipeline

This pipeline has one processing step. It downloads the two exact CSV files, calculates metrics, creates an Evidently report, and writes artifacts to S3 and MLflow.


In [ ]:
param_predictions_s3_uri = ParameterString(
    name='PredictionsS3Uri',
    default_value=predictions_s3_uri,
)

param_ground_truth_s3_uri = ParameterString(
    name='GroundTruthS3Uri',
    default_value=ground_truth_s3_uri,
)

In [ ]:
sklearn_image_uri = image_uris.retrieve(
    framework='sklearn',
    region=region,
    version='1.2-1',
    py_version='py3',
    instance_type=instance_type,
)

model_quality_processor = ScriptProcessor(
    role=role,
    image_uri=sklearn_image_uri,
    command=['python3'],
    instance_count=1,
    instance_type=instance_type,
    volume_size_in_gb=30,
    max_runtime_in_seconds=3600,
    base_job_name='model-quality-monitoring',
    sagemaker_session=pipeline_session,
    env={
        'MLFLOW_TRACKING_URI': mlflow_app_arn,
    },
)

print(f'Processing image: {sklearn_image_uri}')

In [ ]:
processing_args = model_quality_processor.run(
    code='scripts/model_quality_processor.py',
    inputs=[],
    outputs=[
        ProcessingOutput(
            output_name='model-quality-reports',
            s3_output=ProcessingS3Output(
                s3_uri=f's3://{bucket}/{prefix}/monitoring-reports/model-quality',
                local_path='/opt/ml/processing/output',
                s3_upload_mode='EndOfJob',
            ),
        ),
    ],
    arguments=[
        '--predictions-s3-uri', param_predictions_s3_uri,
        '--ground-truth-s3-uri', param_ground_truth_s3_uri,
        '--target-column', target_column,
        '--prediction-column', prediction_column,
        '--prediction-proba-column', prediction_proba_column,
        '--positive-label', positive_label,
        '--mlflow-tracking-uri', mlflow_app_arn,
        '--mlflow-experiment-name', mlflow_experiment_name,
        '--mlflow-run-name', mlflow_run_name,
        '--region', region,
    ],
)

processing_step = ProcessingStep(
    name='ModelQualityMonitoring',
    step_args=processing_args,
)

model_quality_pipeline = Pipeline(
    name=pipeline_name,
    parameters=[param_predictions_s3_uri, param_ground_truth_s3_uri],
    steps=[processing_step],
    sagemaker_session=pipeline_session,
)

model_quality_pipeline.upsert(role_arn=role)
print(f'Pipeline created or updated: {pipeline_name}')

## Section 5: Run the Model Quality Pipeline


In [ ]:
%%time
execution = model_quality_pipeline.start(
    parameters={
        'PredictionsS3Uri': predictions_s3_uri,
        'GroundTruthS3Uri': ground_truth_s3_uri,
    }
)

print(f'Pipeline execution started: {execution.arn}')
print('Waiting for the pipeline execution to finish...')
execution.wait()

status = execution.describe()['PipelineExecutionStatus']
print(f'Pipeline execution status: {status}')

In [ ]:
for step in execution.list_steps():
    print(f'Step: {step["StepName"]}')
    print(f'  Status: {step["StepStatus"]}')
    if 'FailureReason' in step:
        print(f'  Failure reason: {step["FailureReason"]}')

## Section 6: Review Results

Open SageMaker Studio and go to the MLflow app configured above. Look for the run named `model_quality_monitoring_<timestamp>`.

The run contains:

- `Accuracy`
- `Precision`
- `Recall`
- `F1Score`
- `ROC_AUC` when a probability column is available
- Evidently HTML and JSON model quality report artifacts
- `model_quality_summary.json` with the exact S3 files used


## Section 7: Cleanup

This removes the SageMaker Pipeline definition. MLflow runs and S3 artifacts are preserved unless you delete them separately.


In [ ]:
try:
    sm_client.delete_pipeline(PipelineName=pipeline_name)
    print(f'Deleted pipeline: {pipeline_name}')
except Exception as error:
    print(f'Pipeline cleanup skipped or failed: {error}')